# Emotion Recognition in Text using NLP
**Istanbul Medipol University — NLP Course Project**

**Group:** Arya Ghazizadeh (64210017) & Mohammad Shafizadeh (64210053)

> **Instructions:** Runtime → Change runtime type → **T4 GPU**, then run all cells (Ctrl+F9)

---

## 0. Setup & Install Dependencies

In [ ]:
!pip install datasets transformers accelerate wordcloud seaborn scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, json, os
from collections import Counter
from datasets import load_dataset
from wordcloud import WordCloud

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Create output dirs
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/metrics', exist_ok=True)

EMOTION_LABELS = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
label_names = [EMOTION_LABELS[i] for i in range(6)]

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Setup complete!')

## 1. Load Dataset

In [ ]:
dataset = load_dataset('dair-ai/emotion')
print(dataset)

df_train = pd.DataFrame(dataset['train'])
df_val   = pd.DataFrame(dataset['validation'])
df_test  = pd.DataFrame(dataset['test'])

df_train['emotion'] = df_train['label'].map(EMOTION_LABELS)
df_val['emotion']   = df_val['label'].map(EMOTION_LABELS)
df_test['emotion']  = df_test['label'].map(EMOTION_LABELS)

print(f'\nTrain: {len(df_train)}, Validation: {len(df_val)}, Test: {len(df_test)}')
df_train.head(10)

---\n## Part 1: Exploratory Data Analysis (EDA)

### 1.1 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, df) in zip(axes, [('Train', df_train), ('Validation', df_val), ('Test', df_test)]):
    counts = df['emotion'].value_counts().sort_index()
    colors = sns.color_palette('Set2', len(counts))
    ax.bar(counts.index, counts.values, color=colors)
    ax.set_title(f'{name} Set (n={len(df)})')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Emotion Class Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClass proportions (train):')
print(df_train['emotion'].value_counts(normalize=True).round(3).to_string())

### 1.2 Text Length Analysis

In [ ]:
df_train['text_length'] = df_train['text'].str.len()
df_train['word_count'] = df_train['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_train, x='emotion', y='text_length', ax=axes[0], palette='Set2')
axes[0].set_title('Character Length by Emotion')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df_train, x='emotion', y='word_count', ax=axes[1], palette='Set2')
axes[1].set_title('Word Count by Emotion')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('results/figures/text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(df_train.groupby('emotion')[['text_length', 'word_count']].describe().round(1))

### 1.3 Word Clouds per Emotion

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
emotions = sorted(df_train['emotion'].unique())

for ax, emotion in zip(axes.flat, emotions):
    text = ' '.join(df_train[df_train['emotion'] == emotion]['text'].values)
    wc = WordCloud(width=400, height=300, background_color='white', max_words=100, colormap='viridis')
    wc.generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(emotion.upper(), fontsize=14)
    ax.axis('off')

plt.suptitle('Word Clouds by Emotion', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('results/figures/wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 Top 15 Words per Emotion

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, emotion in zip(axes.flat, emotions):
    words = ' '.join(df_train[df_train['emotion'] == emotion]['text']).split()
    common = Counter(words).most_common(15)
    words_list, counts = zip(*common)
    ax.barh(range(len(words_list)), counts, color=sns.color_palette('Set2')[emotions.index(emotion)])
    ax.set_yticks(range(len(words_list)))
    ax.set_yticklabels(words_list)
    ax.invert_yaxis()
    ax.set_title(emotion.upper())

plt.suptitle('Top 15 Words per Emotion', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('results/figures/top_words.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 Dataset Summary

In [ ]:
print('=' * 40)
print('DATASET SUMMARY')
print('=' * 40)
print(f'Total samples: {len(df_train) + len(df_val) + len(df_test)}')
print(f'Emotion classes: {list(EMOTION_LABELS.values())}')
print(f'Avg text length: {df_train["text_length"].mean():.0f} chars, {df_train["word_count"].mean():.0f} words')
print(f'\nClass balance (train):')
for emotion, pct in df_train['emotion'].value_counts(normalize=True).items():
    print(f'  {emotion:10s} {pct:.1%}')

---\n## Part 2: Baseline Model — TF-IDF + Logistic Regression

### 2.1 Preprocess Text

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

X_train_clean = [clean_text(t) for t in dataset['train']['text']]
y_train = dataset['train']['label']

X_val_clean = [clean_text(t) for t in dataset['validation']['text']]
y_val = dataset['validation']['label']

X_test_clean = [clean_text(t) for t in dataset['test']['text']]
y_test_labels = dataset['test']['label']

print(f'Train: {len(X_train_clean)}, Val: {len(X_val_clean)}, Test: {len(X_test_clean)}')
print(f'\nBefore: "{dataset["train"]["text"][0]}"')
print(f'After:  "{X_train_clean[0]}"')

### 2.2 Train with Grid Search

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs', multi_class='multinomial')),
])

param_grid = {'clf__C': [0.1, 1, 5, 10]}
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search.fit(X_train_clean, y_train)

baseline_model = grid_search.best_estimator_
print(f'\nBest C: {grid_search.best_params_["clf__C"]}')
print(f'Best CV F1 (macro): {grid_search.best_score_:.4f}')

### 2.3 Evaluate Baseline on Test Set

In [ ]:
y_pred_baseline = baseline_model.predict(X_test_clean)

print('=' * 50)
print('BASELINE RESULTS: TF-IDF + Logistic Regression')
print('=' * 50)
baseline_report = classification_report(y_test_labels, y_pred_baseline, target_names=label_names, output_dict=True)
print(classification_report(y_test_labels, y_pred_baseline, target_names=label_names))

# Save metrics
with open('results/metrics/baseline_report.json', 'w') as f:
    json.dump(baseline_report, f, indent=2)
print('Metrics saved to results/metrics/baseline_report.json')

### 2.4 Baseline Confusion Matrix

In [ ]:
cm_baseline = confusion_matrix(y_test_labels, y_pred_baseline)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_baseline, display_labels=label_names)
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Baseline: TF-IDF + Logistic Regression', fontsize=13)
plt.tight_layout()
plt.savefig('results/figures/baseline_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Baseline Per-Class F1

In [ ]:
f1_baseline = {e: baseline_report[e]['f1-score'] for e in label_names}

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('Set2', len(f1_baseline))
bars = ax.bar(f1_baseline.keys(), f1_baseline.values(), color=colors)
ax.set_ylabel('F1-Score')
ax.set_title('Per-Class F1 Scores — Baseline Model')
ax.set_ylim(0, 1)
for i, (k, v) in enumerate(f1_baseline.items()):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('results/figures/baseline_f1_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.6 Baseline Error Analysis

In [ ]:
errors = [(X_test_clean[i], label_names[y_test_labels[i]], label_names[y_pred_baseline[i]])
          for i in range(len(X_test_clean)) if y_test_labels[i] != y_pred_baseline[i]]

print(f'Total misclassifications: {len(errors)} / {len(X_test_clean)} ({len(errors)/len(X_test_clean)*100:.1f}%)')
print('\nSample misclassifications:')
print('-' * 90)
for text, true, pred in errors[:15]:
    print(f'  True: {true:10s} | Pred: {pred:10s} | {text[:70]}')

---\n## Part 3: BERT Fine-Tuning

### 3.1 Tokenize Dataset

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import torch

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized = {split: dataset[split].map(tokenize_fn, batched=True) for split in ['train', 'validation', 'test']}

for split in tokenized:
    tokenized[split].set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f'\nTokenization complete.')
print(f'Sample: {tokenizer.decode(tokenized["train"][0]["input_ids"][:20])}')

### 3.2 Fine-Tune BERT

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6)

training_args = TrainingArguments(
    output_dir='./bert-emotion',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none',
    seed=42,
    fp16=torch.cuda.is_available(),  # mixed precision if GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
)

print('Starting BERT training...')
trainer.train()
print('\nTraining complete!')

### 3.3 Evaluate BERT on Test Set

In [ ]:
preds_output = trainer.predict(tokenized['test'])
y_pred_bert = np.argmax(preds_output.predictions, axis=-1)
y_test_arr = np.array(dataset['test']['label'])

print('=' * 50)
print('BERT FINE-TUNED RESULTS')
print('=' * 50)
bert_report = classification_report(y_test_arr, y_pred_bert, target_names=label_names, output_dict=True)
print(classification_report(y_test_arr, y_pred_bert, target_names=label_names))

# Save metrics
with open('results/metrics/bert_report.json', 'w') as f:
    json.dump(bert_report, f, indent=2)
print('Metrics saved to results/metrics/bert_report.json')

### 3.4 BERT Confusion Matrix

In [ ]:
cm_bert = confusion_matrix(y_test_arr, y_pred_bert)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_bert, display_labels=label_names)
disp.plot(cmap='Oranges', ax=ax, values_format='d')
ax.set_title('BERT Fine-Tuned — Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('results/figures/bert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Training Loss Curve

In [ ]:
log_history = trainer.state.log_history
train_loss = [(e['step'], e['loss']) for e in log_history if 'loss' in e]
eval_loss = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(10, 4))
if train_loss:
    steps, losses = zip(*train_loss)
    ax.plot(steps, losses, label='Train Loss', alpha=0.7)
if eval_loss:
    steps, losses = zip(*eval_loss)
    ax.plot(steps, losses, label='Eval Loss', marker='o', markersize=6)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('BERT Training & Evaluation Loss')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/bert_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---\n## Part 4: Comparison — Baseline vs BERT

### 4.1 Side-by-Side Metrics Table

In [ ]:
comparison = pd.DataFrame({
    'Baseline (TF-IDF+LogReg)': {e: baseline_report[e]['f1-score'] for e in label_names},
    'BERT (fine-tuned)': {e: bert_report[e]['f1-score'] for e in label_names},
})
comparison['Improvement'] = comparison['BERT (fine-tuned)'] - comparison['Baseline (TF-IDF+LogReg)']

print('=' * 60)
print('PER-CLASS F1-SCORE COMPARISON')
print('=' * 60)
print(comparison.round(4).to_string())
print(f'\n{"="*60}')
print(f'Overall Accuracy  — Baseline: {baseline_report["accuracy"]:.4f}  |  BERT: {bert_report["accuracy"]:.4f}')
print(f'Macro F1          — Baseline: {baseline_report["macro avg"]["f1-score"]:.4f}  |  BERT: {bert_report["macro avg"]["f1-score"]:.4f}')
print(f'Weighted F1       — Baseline: {baseline_report["weighted avg"]["f1-score"]:.4f}  |  BERT: {bert_report["weighted avg"]["f1-score"]:.4f}')

# Save comparison
comparison.round(4).to_csv('results/metrics/comparison.csv')

### 4.2 F1 Comparison Chart

In [ ]:
x = np.arange(len(label_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, [baseline_report[e]['f1-score'] for e in label_names], width,
               label='TF-IDF + LogReg', color='#66c2a5', edgecolor='white')
bars2 = ax.bar(x + width/2, [bert_report[e]['f1-score'] for e in label_names], width,
               label='BERT (fine-tuned)', color='#fc8d62', edgecolor='white')

ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('Per-Class F1: Baseline vs BERT', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([e.capitalize() for e in label_names], fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/comparison_f1.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Confusion Matrices Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ConfusionMatrixDisplay(cm_baseline, display_labels=label_names).plot(cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title('Baseline: TF-IDF + LogReg', fontsize=13)

ConfusionMatrixDisplay(cm_bert, display_labels=label_names).plot(cmap='Oranges', ax=axes[1], values_format='d')
axes[1].set_title('BERT Fine-Tuned', fontsize=13)

plt.suptitle('Confusion Matrix Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('results/figures/comparison_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Overall Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Weighted F1'],
    'Baseline': [
        baseline_report['accuracy'],
        baseline_report['macro avg']['precision'],
        baseline_report['macro avg']['recall'],
        baseline_report['macro avg']['f1-score'],
        baseline_report['weighted avg']['f1-score'],
    ],
    'BERT': [
        bert_report['accuracy'],
        bert_report['macro avg']['precision'],
        bert_report['macro avg']['recall'],
        bert_report['macro avg']['f1-score'],
        bert_report['weighted avg']['f1-score'],
    ],
})
summary['Improvement'] = summary['BERT'] - summary['Baseline']
print(summary.round(4).to_string(index=False))

# Save
summary.round(4).to_csv('results/metrics/overall_summary.csv', index=False)
print('\nAll results saved to results/ folder!')

### 4.5 Download All Results

In [ ]:
# Zip all results for download
!zip -r results.zip results/
from google.colab import files
files.download('results.zip')
print('Download started! Check your browser downloads.')

---
## Done!

All figures are in `results/figures/` and all metrics in `results/metrics/`.

**Figures generated:**
- `class_distribution.png` — EDA
- `text_length_analysis.png` — EDA
- `wordclouds.png` — EDA
- `top_words.png` — EDA
- `baseline_confusion_matrix.png` — Baseline
- `baseline_f1_per_class.png` — Baseline
- `bert_confusion_matrix.png` — BERT
- `bert_training_curve.png` — BERT
- `comparison_f1.png` — Comparison
- `comparison_confusion_matrices.png` — Comparison

**Metrics saved:**
- `baseline_report.json`
- `bert_report.json`
- `comparison.csv`
- `overall_summary.csv`
